# Estimate Compute And Storage

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
from scientific_figures import generate_figures,ROOT,read
generate_figures()
print(json.dumps(read(ROOT/'compute_estimate.json'),indent=2))
print('Estimate Compute And Storage definitions/execution completed.')


Frozen runtime contract definitions/execution completed.


Scientific figures and conditional compute estimate definitions/execution completed.


{
  "status": "CONDITIONAL_PLANNING_ESTIMATE",
  "measured_training_rates": {
    "TASK": 106.97672874470288,
    "MAX": 119.18945698093823
  },
  "evaluation_worker_seconds_per_episode": {
    "TASK": 8.730315394299396,
    "MAX": 8.71642530310055,
    "RANDOM": 7.157392631999391
  },
  "evaluation_loop_seconds_per_episode": {
    "TASK": 4.498388764700212,
    "MAX": 4.77406592970001,
    "RANDOM": 3.7952872505004054
  },
  "evaluation_nonloop_seconds_per_episode": {
    "TASK": 4.231926629599184,
    "MAX": 3.9423593734005404,
    "RANDOM": 3.362105381498986
  },
  "training_nonloop_seconds_per_policy": {
    "TASK": 7.160639515001094,
    "MAX": 5.823038205002376
  },
  "nominal": {
    "decisions_per_policy": 1000000,
    "core_decisions": 25000000,
    "training_loop_seconds_range": [
      214539.2136863265,
      228906.56785126778
    ],
    "training_nonloop_seconds_range": [
      145.5759551250594,
      179.01598787502735
    ],
    "evaluation_seconds_range": [
      6754